# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [8]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [9]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR 
PROCESSED_DATA_DIR = PROJECT_DIR / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [10]:
train_df = pd.read_csv(PROCESSED_DATA_DIR /"train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [11]:
feature_columns = contract["features"]

# Preprocesador para árboles: solo imputación por mediana (no necesitan escala)
tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), feature_columns)
])

# Preprocesador para modelos sensibles a la escala: imputación + estandarización
scaled_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), feature_columns)
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si se ajusta el `StandardScaler` sobre todos los datos antes de la validación cruzada, la media y la desviación usadas para escalar incluyen información de los folds de validación, lo que produce *data leakage* y métricas optimistas. Al ponerlo dentro del pipeline, en cada fold el scaler se ajusta solo con los datos de entrenamiento de ese fold y luego transforma el de validación, igual que ocurriría en producción con datos nuevos.

## 3. Modelos candidatos

In [12]:
# Pipelines: cada modelo lleva su preprocesador adecuado
models = {
    "decision_tree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "logistic_regression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "knn": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("model", KNeighborsClassifier())
    ]),
}

## 4. Comparación inicial con validación cruzada

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

initial_comparison = {}
for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    initial_comparison[name] = {
        "mean_train_accuracy": scores["train_score"].mean(),
        "mean_val_accuracy": scores["test_score"].mean(),
        "std_val_accuracy": scores["test_score"].std(),
    }

initial_comparison = pd.DataFrame(initial_comparison).T
print(initial_comparison)

                     mean_train_accuracy  mean_val_accuracy  std_val_accuracy
decision_tree                     1.0000           0.902217          0.058617
logistic_regression               1.0000           0.979064          0.027713
knn                               0.9771           0.950739          0.035274


## 5. Optimización de hiperparámetros

In [14]:
# Espacios pequeños de hiperparámetros (los nombres usan el prefijo 'model__')
param_grids = {
    "decision_tree": {
        "model__max_depth": [3, 5, None],
        "model__min_samples_leaf": [1, 3],
    },
    "logistic_regression": {
        "model__C": [0.1, 1.0, 10.0],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 7],
    },
}

searches = {}
tuned_comparison = {}
for name, pipeline in models.items():
    search = GridSearchCV(
        pipeline,
        param_grids[name],
        scoring="accuracy",
        cv=cv,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    tuned_comparison[name] = {
        "best_score": search.best_score_,
        "best_params": search.best_params_,
    }

tuned_comparison = pd.DataFrame(tuned_comparison).T
print(tuned_comparison)

                    best_score  \
decision_tree         0.902217   
logistic_regression   0.979064   
knn                   0.957882   

                                                           best_params  
decision_tree        {'model__max_depth': 5, 'model__min_samples_le...  
logistic_regression                                  {'model__C': 1.0}  
knn                                          {'model__n_neighbors': 7}  


## 6. Selección y persistencia del campeón

In [15]:
# Selecciona el modelo con mayor best_score_ en la validación cruzada
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_

print("Campeón:", champion_name)
print("Mejor accuracy CV:", searches[champion_name].best_score_)

# Guarda el pipeline campeón completo
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# Metadatos del entrenamiento
training_metadata = {
    "champion": champion_name,
    "champion_name": champion_name,
    "best_params": searches[champion_name].best_params_,
    "best_cv_accuracy": searches[champion_name].best_score_,
    "random_state": RANDOM_STATE,
    "features": feature_columns,
}
with open(ARTIFACTS_DIR / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2)

# Resultados de la validación cruzada
tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv")

Campeón: logistic_regression
Mejor accuracy CV: 0.9790640394088669


In [16]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se elige como campeón el modelo con mayor accuracy medio en la validación cruzada (`best_score_`), ya que es una estimación honesta del desempeño sin usar el conjunto de test. Además del promedio conviene mirar la desviación estándar entre folds (menor variabilidad es más fiable) y la brecha entre train y validation accuracy en `initial_comparison` (una brecha grande indica sobreajuste). Entre modelos de desempeño similar, suele preferirse el más simple e interpretable (por ejemplo, un árbol poco profundo o la regresión logística frente a KNN). El resultado numérico concreto depende de ejecutar el notebook con los artefactos del Challenge 1.

